## Step 1: Install and imports

In [ ]:
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn

## Step 2: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from google.colab import files

## Step 3: Upload CSVs

Upload `cnn_embeddings_daily.csv` and `coral_climatebert_gbr_embeddings.csv`

In [ ]:
uploaded = files.upload()

cnn_df     = pd.read_csv("cnn_embeddings_daily.csv")
climate_df = pd.read_csv("coral_climatebert_gbr_embeddings.csv")

cnn_df["date"] = pd.to_datetime(cnn_df["date"])

print("CNN shape      :", cnn_df.shape)
print("ClimateBERT shape:", climate_df.shape)
print("Unique seasons :", sorted(climate_df["season_key"].unique()))

## Step 4: Remap labels and map CNN days to seasons

In [ ]:
# Remap labels: 5 -> 3 classes
LABEL_MAP = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2}
cnn_df["label"] = cnn_df["label"].map(LABEL_MAP)
print("Remapped label distribution:")
print(cnn_df["label"].value_counts().sort_index())

# Season helpers (Australian seasons)
def get_season(month):
    if month in [12, 1, 2]:  return "Summer"
    elif month in [3, 4, 5]: return "Autumn"
    elif month in [6, 7, 8]: return "Winter"
    else:                     return "Spring"

def get_season_year(date):
    season = get_season(date.month)
    year   = date.year - 1 if (season == "Summer" and date.month == 12) else date.year
    return season, year

cnn_df["season"], cnn_df["season_year"] = zip(*cnn_df["date"].apply(get_season_year))
cnn_df["season_key"] = cnn_df["season_year"].astype(str) + "-" + cnn_df["season"]

# ClimateBERT embeddings
cb_emb_cols = [c for c in climate_df.columns if c.startswith("emb_")]
climate_lookup = climate_df.set_index("season_key")[cb_emb_cols]
climate_lookup.columns = [f"cb_{c}" for c in cb_emb_cols]

# Broadcast seasonal ClimateBERT vector onto every CNN daily row
cnn_df = cnn_df.merge(climate_lookup, on="season_key", how="left")
cb_ctx_cols = [c for c in cnn_df.columns if c.startswith("cb_")]
cnn_df[cb_ctx_cols] = cnn_df[cb_ctx_cols].fillna(0.0)

print(f"\nDays with ClimateBERT vector: {(cnn_df[cb_ctx_cols[0]] != 0.0).sum()} / {len(cnn_df)}")
print(f"ClimateBERT dims: {len(cb_ctx_cols)}")

## Step 5: Normalise embeddings

In [ ]:
CNN_DIM = 256
CB_DIM  = len(cb_ctx_cols)

cnn_emb_cols = [f"emb_{i}" for i in range(CNN_DIM)]
train_mask   = cnn_df["split"] == "train"
test_mask    = cnn_df["split"] == "test"

scaler_cnn = StandardScaler()
scaler_cb  = StandardScaler()

cnn_df.loc[train_mask, cnn_emb_cols] = scaler_cnn.fit_transform(cnn_df.loc[train_mask, cnn_emb_cols])
cnn_df.loc[test_mask,  cnn_emb_cols] = scaler_cnn.transform(cnn_df.loc[test_mask,  cnn_emb_cols])

cnn_df.loc[train_mask, cb_ctx_cols] = scaler_cb.fit_transform(cnn_df.loc[train_mask, cb_ctx_cols])
cnn_df.loc[test_mask,  cb_ctx_cols] = scaler_cb.transform(cnn_df.loc[test_mask,  cb_ctx_cols])

print(f"Normalised. CNN: {CNN_DIM}-dim  ClimateBERT: {CB_DIM}-dim  Total: {CNN_DIM + CB_DIM}-dim")

## Step 6: Build sequences

In [ ]:
SEQ_LEN = 30

train_df = cnn_df[cnn_df["split"] == "train"].sort_values("date").reset_index(drop=True)
test_df  = cnn_df[cnn_df["split"] == "test"].sort_values("date").reset_index(drop=True)

def build_sequences(df, seq_len):
    X_cnn, X_cb, y = [], [], []
    for i in range(len(df) - seq_len + 1):
        window = df.iloc[i:i+seq_len]
        X_cnn.append(window[cnn_emb_cols].values)
        X_cb.append(window[cb_ctx_cols].values[-1])
        y.append(window["label"].values[-1])
    return (
        np.array(X_cnn, dtype=np.float32),
        np.array(X_cb,  dtype=np.float32),
        np.array(y,     dtype=np.int64)
    )

X_cnn_train, X_cb_train, y_train = build_sequences(train_df, SEQ_LEN)
X_cnn_test,  X_cb_test,  y_test  = build_sequences(test_df,  SEQ_LEN)

print(f"Train: CNN {X_cnn_train.shape}  CB {X_cb_train.shape}  Labels {y_train.shape}")
print(f"Test : CNN {X_cnn_test.shape}   CB {X_cb_test.shape}   Labels {y_test.shape}")

## Step 7: Dataset and DataLoader

In [ ]:
class SeqFusionDataset(Dataset):
    def __init__(self, X_cnn, X_cb, y):
        self.X_cnn = torch.tensor(X_cnn)
        self.X_cb  = torch.tensor(X_cb)
        self.y     = torch.tensor(y)

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        return self.X_cnn[idx], self.X_cb[idx], self.y[idx]

train_ds = SeqFusionDataset(X_cnn_train, X_cb_train, y_train)
test_ds  = SeqFusionDataset(X_cnn_test,  X_cb_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f"Train: {len(train_ds)}  |  Test: {len(test_ds)}")

## Step 8: Cross-attention fusion model

In [ ]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, cnn_dim=256, cb_dim=768, hidden_dim=256,
                 num_heads=4, num_classes=3, dropout=0.3):
        super().__init__()

        self.cb_proj = nn.Sequential(
            nn.Linear(cb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        self.cnn_proj   = nn.Linear(cnn_dim, hidden_dim)
        self.pos_enc    = nn.Embedding(100, hidden_dim)
        self.self_attn  = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, cnn_seq, cb_vec):
        B, T, _ = cnn_seq.shape
        cnn_h   = self.cnn_proj(cnn_seq)
        positions = torch.arange(T, device=cnn_seq.device).unsqueeze(0)
        cnn_h   = cnn_h + self.pos_enc(positions)
        cnn_h   = self.self_attn(cnn_h)
        cb_h    = self.cb_proj(cb_vec).unsqueeze(1)
        fused, _ = self.cross_attn(query=cnn_h, key=cb_h, value=cb_h)
        fused   = self.norm(fused + cnn_h)
        out     = fused[:, -1, :]
        return self.classifier(out)

## Step 9: Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

label_counts  = np.bincount(y_train)
class_weights = torch.tensor(
    1.0 / label_counts * label_counts.sum() / len(label_counts),
    dtype=torch.float32
).to(device)
print("Class weights:", class_weights.cpu().numpy().round(3))

model     = CrossAttentionFusion(cb_dim=CB_DIM).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS, PATIENCE = 100, 15
best_val_acc, patience_ctr, best_state = 0.0, 0, None
train_losses, val_accs = [], []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for cnn_b, cb_b, labels in train_loader:
        cnn_b, cb_b, labels = cnn_b.to(device), cb_b.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(cnn_b, cb_b)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for cnn_b, cb_b, labels in test_loader:
            cnn_b, cb_b, labels = cnn_b.to(device), cb_b.to(device), labels.to(device)
            logits = model(cnn_b, cb_b)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)

    acc = correct / total
    train_losses.append(total_loss / len(train_loader))
    val_accs.append(acc)

    if acc > best_val_acc:
        best_val_acc = acc
        patience_ctr = 0
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {train_losses[-1]:.4f} | Val Acc: {acc:.4f} | Best: {best_val_acc:.4f} | Patience: {patience_ctr}/{PATIENCE}")

    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}. Best val acc: {best_val_acc:.4f}")
        break

model.load_state_dict(best_state)
print(f"\nRestored best model (val acc: {best_val_acc:.4f})")

## Step 10: Evaluate

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for cnn_b, cb_b, labels in test_loader:
        cnn_b, cb_b = cnn_b.to(device), cb_b.to(device)
        logits = model(cnn_b, cb_b)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print("=== ClimateBERT Cross-Attention Fusion ===")
print(classification_report(all_labels, all_preds,
      target_names=["No Stress", "Moderate", "Severe"]))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Stress","Moderate","Severe"],
            yticklabels=["No Stress","Moderate","Severe"])
plt.title("Confusion Matrix — ClimateBERT Cross-Attention Fusion")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/content/confusion_matrix_climatebert.png", dpi=150)
plt.show()

## Step 11: Training curves

In [ ]:
best_epoch = int(np.argmax(val_accs))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses)
ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax2.plot(val_accs, label="Val Accuracy")
ax2.axvline(best_epoch, color="red", linestyle="--",
            label=f"Best epoch {best_epoch+1} ({best_val_acc:.3f})")
ax2.set_title("Validation Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()
plt.tight_layout()
plt.savefig("/content/training_curves_climatebert.png", dpi=150)
plt.show()
print(f"Best epoch: {best_epoch+1}  |  Best val acc: {best_val_acc:.4f}")

## Step 12: Save and download

In [ ]:
torch.save(model.state_dict(), "/content/climatebert_crossattn_fusion.pth")
from google.colab import files as colab_files
colab_files.download("/content/climatebert_crossattn_fusion.pth")
colab_files.download("/content/confusion_matrix_climatebert.png")
colab_files.download("/content/training_curves_climatebert.png")
print("Done.")